In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
from statistics import mean
import matplotlib.pyplot as plt
import warnings
from sklearn.preprocessing import PowerTransformer
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.utils import resample

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report,ConfusionMatrixDisplay, \
                            precision_score, recall_score, f1_score, roc_auc_score,roc_curve,confusion_matrix


from sklearn import metrics 
from sklearn.model_selection import  train_test_split, RepeatedStratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler,RobustScaler
from sklearn.compose import ColumnTransformer
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
%matplotlib inline

In [ ]:
df = pd.read_csv(r'aps_failure_training_set1.csv',na_values="na")

In [ ]:
df

In [ ]:
df.shape

In [ ]:
df['class'].value_counts()

In [ ]:
#we are categorizing data in tabular forms and differentiating it in class and numerical features

numeric_features = [feature for feature in df.columns if df[feature].dtype != 'O']
categorical_features = [feature for feature in df.columns if df[feature].dtype == 'O']

In [ ]:
print('We have {} numerical features : {}'.format(len(numeric_features), numeric_features))
print('\nWe have {} categorical features : {}'.format(len(categorical_features), categorical_features))

In [ ]:
#checking missing values

fig, ax = plt.subplots(figsize=(15,5))

missing = df.isna().sum().div(df.shape[0]).mul(100).to_frame().sort_values(by=0, ascending = False)

ax.bar(missing.index, missing.values.T[0])
plt.xticks([])
plt.ylabel("Percentage missing")
plt.show()

In [ ]:
#drop columns containg more than 70% missing values

dropcols = missing[missing[0]>70]
dropcols

In [ ]:
df.drop(list(dropcols.index),axis=1,inplace=True)

In [ ]:
#now check shape of the data
df.shape

In [ ]:
missing_values_count = df.isnull().sum()
total_cells = np.product(df.shape)
total_missing = missing_values_count.sum()

#Now check the percentage of the missing data

print(f"Percentage of total missing cells in the data {(total_missing/total_cells) * 100}%")


In [ ]:
#here we are visualizing unique values and differentiating neg and pos

pos = df[df['class']=='pos'].shape[0]
neg = df[df['class']=='neg'].shape[0]
print("Positive: " + str(pos) + ", Negative: " +str(neg))
sns.catplot(data=df, x="class", kind= "count", palette = "winter_r", alpha=.6)
plt.show()

In [ ]:
def evaluate_clf(true, predicted):
    '''This function is used in returning Accuracy , F1-score,Precision,Recall ,Roc-auc Score 
    using true values and predicted values'''
    acc = accuracy_score(true, predicted)
    f1 = f1_score(true, predicted)
    precision = precision_score(true, predicted)
    recall = recall_score(true,predicted)
    roc_auc = roc_auc_score(true, predicted)
    return acc,f1,precision,recall,roc_auc

In [ ]:
def total_cost(y_true,y_pred):
    ''' 
    This function takes y_ture, y_predicted and prints total cost due to missclassification
    '''
    tn,fp,fn,tp = confusion_matrix(y_true, y_pred).ravel()
    cost = 10*fp+500*fn
    return cost

In [ ]:
def evaluate_models(X,y,models):
    '''This function takes in X and y and models
    dictionary as input . Its splits the data into Train
    test split . Iterates through the given model dictionary and evalutes the metrics
    Returns: Dataframe which contains report of all models metrics with cost'''
    X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=42)
    
    cost_list = []
    models_list = []
    accuracy_list = []
    
    for i in range(len(list(models))):
        model = list(models.values())[i]
        model.fit(X_train, y_train) #Train model
        
        #Make predictions
        
        y_train_pred = model.predict(X_train)
        
        y_test_pred = model.predict(X_test)
        
        # Training Set performance
        model_train_accuracy,model_train_f1,model_train_precision,\
        model_test_recall,model_train_rocauc_score=evaluate_clf(y_test,y_test_pred)
        train_cost = total_cost(y_test, y_test_pred)
        
        #Testing set performance
        model_test_accuracy, model_test_f1, model_test_precision,\
        model_test_recall,model_test_rocauc_score=evaluate_clf(y_train , y_train_pred)
        test_cost=total_cost(y_train,y_train_pred)
        
        print(list(models.keys())[i])
        models_list.append(list(models.keys())[i])
        
        print('Model performance for Training set')
        print("- Accuracy: {:.4f}".format(model_train_accuracy))
        print("- F1 score: {:.4f}".format(model_train_f1))
        print('- Precision: {:.4f}'.format(model_train_precision))
        print('- Recall: {:.4f}'.format(model_train_rocauc_score))
        print(f'-COST: {train_cost}.')
        
        print('--------------------')
        
        print('Model performance for Test set')
        print('-Accuracy: {:.4f}'.format(model_test_accuracy))
        print('-F1 score : {:.4f}'.format(model_test_f1))
        print('- Precision: {:.4f}'.format(model_test_precision))
        print('-Recall: {:.4f}'.format(model_test_recall))
        print('- ROC Auc Score : {:.4f}'.format(model_test_rocauc_score))
        cost_list.append(test_cost)
        print('='*35)
        print('\n')
    report = pd.DataFrame(list(zip(models_list, cost_list)), columns=['Model Name ', 'Cost']).sort_values(by=["Cost"])
    return report

In [ ]:
#Splitting x and y for all Experiments
X = df.drop('class', axis=1)
y = df['class']

In [ ]:
y = y.replace({'pos':1,'neg':0})

In [ ]:
robustscaler = RobustScaler()
X1 = robustscaler.fit_transform(X)

In [ ]:
results=[]
# define imputer
imputer = KNNImputer(n_neighbors=5, weights='uniform', metric='nan_euclidean')
strategies = [str(i) for i in [1,3,5,7,9]]
for s in strategies:
    pipeline = Pipeline(steps=[('i', KNNImputer(n_neighbors=int(s))), ('m', LogisticRegression())])
    scores = cross_val_score(pipeline, X1, y, scoring='accuracy', cv=2, n_jobs=-1)
    results.append(scores)
    print('n_neighbors= %s || accuracy (%.4f)' % (s , mean(scores)))

In [ ]:
num_features = X.select_dtypes(exclude="object").columns
#combine KNN imputer with selected k-value
knn_pipeline = Pipeline(
    steps=[
        ('imputer',KNNImputer(n_neighbors=3)),
        ('RobustScaler', RobustScaler())
    ]
)

In [ ]:
x_knn = knn_pipeline.fit_transform(X)

In [ ]:
from imblearn.combine import SMOTETomek

#Resampling these minority class can help in creating strategy that can be changed as required
smt = SMOTETomek(random_state=42,sampling_strategy='minority',n_jobs=-1)
#Fit this model in order to generate the data
X_res, y_res = smt.fit_resample(x_knn,y)

In [ ]:
# Dictionary which contains models for experiment
models = {
    "Random Forest": RandomForestClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "Logistic Regression": LogisticRegression(),
     "K-Neighbors Classifier": KNeighborsClassifier(),
    "XGBClassifier": XGBClassifier(), 
     "CatBoosting Classifier": CatBoostClassifier(verbose=False),
    "AdaBoost Classifier": AdaBoostClassifier()
}

In [ ]:
report_knn = evaluate_models(X_res, y_res, models)

In [ ]:
report_knn

In [ ]:
num_features = X.select_dtypes(exclude="object").columns

#Fitting this simple imputer with strategy median
median_pipeline = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='median')),
        ('RobustScaler',RobustScaler())
    ]
)

In [ ]:
#Fit X with median_pipeline
X_median = median_pipeline.fit_transform(X)

In [ ]:
#Resampling this minority class . the strategy can be changed as per requirements
smt = SMOTETomek(random_state = 42, sampling_strategy='minority')
#Fit the model to generate the data
X_res, y_res = smt.fit_resample(X_median, y)

In [ ]:
#Now Train these models
report_median = evaluate_models(X_res, y_res, models)

In [ ]:
report_median

In [ ]:
import miceforest as mf

X_mice = X.copy()
kernel = mf.ImputationKernel(
    X_mice,
    random_state = 1989
)#Run the MICE Algorithm for 3 iterations kernel.mice

In [ ]:
X_mice - kernel.complete_data()

In [ ]:
#Fit robust scaler
mice_pipeline = Pipeline(
    steps=[
        ('RobustScaler',RobustScaler())
    ]
)

In [ ]:
#fit X with Mice imputer
X_mice = mice_pipeline.fit_transform(X_mice)

In [ ]:
print(X.isnull().sum())

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='mean')
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

In [ ]:
kernel = mf.ImputationKernel(X)
X = kernel.complete_data()

In [ ]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_resampled, y_resampled = smt.fit_resample(X,y)

In [ ]:
X = X.dropna()
y = y.loc[X.index]
print(X.isnull().sum().sum())
print(np.isinf(X).sum().sum())  # Should print 0
print(np.isnan(X).sum().sum())  # Should print 0
X = X.astype(float)  # Ensure all values are float
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_resampled, y_resampled = smt.fit_resample(X, y)

print("Resampling successful!")

import pandas as pd
import numpy as np

# Ensure X_mice is a DataFrame
if isinstance(X_mice, np.ndarray):
    X_mice = pd.DataFrame(X_mice)

# Now check for NaN values
print("Total NaN values in X_mice:", X_mice.isnull().sum().sum())  # Should be 0


In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")  # Try "median" if needed
X_mice = imputer.fit_transform(X_mice)  # Keeps X as a NumPy array
X_mice = X_mice.astype(float)
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42, sampling_strategy='minority', n_jobs=-1)
X_res, y_res = smt.fit_resample(X_mice, y)

print("SMOTETomek applied successfully!")


In [ ]:
X = X.dropna()
y = y.loc[X.index]
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")  # Use "median" or "most_frequent" if needed
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

print(X.isnull().sum().sum())
imputer = SimpleImputer(strategy='mean')
X.replace([np.inf, -np.inf], np.nan, inplace=True)  # Convert inf to NaN
X.fillna(X.mean(), inplace=True)  # Impute again

X = X.astype(float)  # Ensure all values are float

X.replace([np.inf, -np.inf], np.nan, inplace=True)  # Convert inf to NaN
X.fillna(X.mean(), inplace=True)  # Impute again

#Resampling the minority class . The strategy can be changed as required
smt = SMOTETomek(random_state=42,sampling_strategy='minority',n_jobs=-1)
#Fit the model to generate the data
X_res, y_res = smt.fit_resample(X_mice, y)

In [ ]:
#Training the models
report_mice = evaluate_models(X_res, y_res, models)

In [ ]:
report_mice

In [ ]:
#Create a pipeline with simple imputer with strategy constant and fill vale 0
constant_pipeline = Pipeline(steps=[
    ('Imputer', SimpleImputer(strategy='constant,fill_value=0')),
    ('RobustScaler',RobustScaler())
])

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="constant", fill_value=0)  
X_const = imputer.fit_transform(X)  


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

constant_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="constant", fill_value=0))  
])

X_const = constant_pipeline.fit_transform(X)  


In [ ]:
X_const = constant_pipeline.fit_transform(X)

In [ ]:
#Resampling the minority class. The strategy can be changed as required
smt = SMOTETomek(random_state=42,sampling_strategy='minority',n_jobs=-1)
#Fit the model to generate the data
X_res, y_res=smt.fit_resample(X_const, y)

In [ ]:
#training the models
report_const = evaluate_models(X_res, y_res,models)

In [ ]:
report_const

In [ ]:
#Create a pipeline with Simple imputer with strategy mean
mean_pipeline = Pipeline(steps=[
    ('Imputer',SimpleImputer(strategy='mean')),
    ('RobustScaler',RobustScaler())
])

In [ ]:
X_mean = mean_pipeline.fit_transform(X)

In [ ]:
#Resampling the minority class. The strategy can be changed as required
smt = SMOTETomek(random_state=42,sampling_strategy='minority',n_jobs=-1)
#Fit the model to generate the data
X_res , y_res = smt.fit_resample(X_mean, y)

In [ ]:
#Training all models
report_mean = evaluate_models(X_res, y_res , models)

In [ ]:
report_mean

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pca_pipeline = Pipeline(
    steps=[
        ('Imputer',SimpleImputer(strategy='constant',fill_value=0)),
        ('RobustScaler',RobustScaler())
    ]
)

In [ ]:
X_pca = pca_pipeline.fit_transform(X)

In [ ]:
#Applyling pca here to show the machine learning algorithm in generic way
from sklearn.decomposition import PCA
var_ratio={}
for n in range(2,150):
    pc = PCA(n_components=n)
    df_pca=pc.fit(X_pca)
    var_ratio[n]=sum(df_pca.explained_variance_ratio_)

In [ ]:
from kneed import KneeLocator

i = np.arange(len(var_ratio))
variance_ratio = list(var_ratio.values())
components = list(var_ratio.keys())
knee = KneeLocator(i, variance_ratio, curve='concave',direction='increasing', interp_method='polynomial')

fig = plt.figure(figsize=(5,5))
knee.plot_knee()
plt.xlabel("Points")
plt.ylabel("Distance")
plt.show()
k=components[knee.knee] if knee.knee is not None else None
print('Knee Locator K=', k)


In [ ]:
#reducing the dimensions of the data 
pca_final = PCA(n_components=18,random_state=42).fit(X_res)

reduced = pca_final.fit_transform(X_pca)

In [ ]:
#Resampling the minority class. The strategy can be changed as required.
smt = SMOTETomek(random_state=42,sampling_strategy='minority',n_jobs=-1)
#Fit the model to generate the data
X_res, y_res = smt.fit_resample(reduced,y)

In [ ]:
#Training all models 
report_pca = evaluate_models(X_res,y_res,models)

In [ ]:
report_pca

In [ ]:
from prettytable import PrettyTable

pt = PrettyTable()
pt.field_names = ["Model","Imputation_method","Total_cost"]
pt.add_row(["XGBClassifier","Simple Imputer-Constant","2950"])
pt.add_row(["XGBClassifier","Mice","3510"])
pt.add_row(["XGBClassifier","Knn-Imputer","4460"])
pt.add_row(["XGBClassifier","Simple Imputer-Mean","4950"])
pt.add_row(["CatBoostClassifier","Median","5760"])
pt.add_row(["Random Forest","PCA","34150"])
print(pt)

In [ ]:
final_model = XGBClassifier

#Resampling the minority class. The Strategy can be changed as required.
smt = SMOTETomek(random_state=42,sampling_strategy='minority',n_jobs=-1)
#Fit the model to generate the data
X_res, y_res = smt.fit_resample(X_const, y)

In [ ]:
from xgboost import XGBClassifier

final_model = XGBClassifier(random_state=42)  # or your custom params
final_model.fit(X_train, y_train)

X_train, X_test, y_train, y_test = train_test_split(X_res,y_res,test_size=0.2,random_state=42)
final_model = final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)

In [ ]:
print("Final XGBoost Classifier Accuracy Score(Train):",final_model.score(X_train,y_train))
print("Final XGBoost Classifier Accuracy Score (Test):",accuracy_score(y_pred,y_test))

In [ ]:
print("Final XGBoost Classifier Cost Metric(Test) :",total_cost(y_test,y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, final_model.predict(X_test))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues', values_format='d')
plt.show()
